In [ ]:
!pip install tensorflow tensorflow-hub

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import sentencepiece

from tqdm.auto import tqdm

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, f1_score, classification_report


/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


### **1. Carregamento dos dados**

In [ ]:
train = pd.read_csv("train.csv")

print(train.shape)
train.head()

X = train["resp_text"]
y = train["clarity"]

print("Número de textos:", len(X))
print("Número de rótulos:", len(y))

(20092, 2)
Número de textos: 20092
Número de rótulos: 20092


### **2. Divisão do conjunto em treino e validação/teste**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=100
)

print("Treino:", X_train.shape)
print("Validação:", X_val.shape)

print("\nClasses no treino:")
print(y_train.value_counts())

print("\nClasses na validação:")
print(y_val.value_counts())

Treino: (16073,)
Validação: (4019,)

Classes no treino:
clarity
c5      5513
c234    5482
c1      5078
Name: count, dtype: int64

Classes na validação:
clarity
c5      1379
c234    1371
c1      1269
Name: count, dtype: int64


### **3. Carregamento do modelo USE**

In [ ]:
import tensorflow_text as text

print("TensorFlow:", tf.__version__)
print("TensorFlow Hub:", hub.__version__)
print("TensorFlow Text:", text.__version__)

TensorFlow: 2.20.0
TensorFlow Hub: 0.16.1
TensorFlow Text: 2.20.1


In [ ]:
model_use = hub.load(
    "https://tfhub.dev/google/universal-sentence-encoder-multilingual/3"
)

print("USE carregado!")

USE carregado!


In [ ]:
texto_exemplo = X_train.iloc[0]
print(texto_exemplo)

embedding_exemplo = model_use([texto_exemplo])
print("Tipo:", type(embedding_exemplo))
print("Shape:", embedding_exemplo.shape)
print(embedding_exemplo[0][:10])

 Prezada Senhora,  Esclarecemos que conforme preceito da Lei 12.527 de 18 de novembro de 2011-Lei de Acesso à Informação- LAI, o Serviço de Informação ao Cidadão- SIC não é instrumento hábil para o recebimento e processamento de reclamações, elogios, denúncias ou resolução de problemas, e-sic é destinado apenas para acesso a informações de interesse público, sendo vedada para este canal, informações de cunho particular.  Face ao exposto, a demanda de n. 23480023370201711 não poderá ser atendida por meio deste canal. Por oportuno, orientamos que para informações particulares, utilize a Central de Atendimento ao Cidadão do FNDE através do telefone 0800-616161 ou o link FALE CONOSCO, localizado no Portal do FNDE:  www.fnde.gov.br, a fim de que sua demanda seja prontamente atendida pela equipe técnica correspondente.  Passo a passo para abertura de demandas:  1 - Acessar o Portal FNDE (www.fnde.gov.br) na barra de endereço do seu navegador, clicar no link Fale Conosco, 2 - Inserir na tel

### **4. Função para gerar todos os embeddings em lotes**

In [ ]:
def gerar_embeddings_use(textos, model, batch_size=32):
    embeddings = []

    textos = list(textos)

    for i in tqdm(
        range(0, len(textos), batch_size),
        desc="Gerando embeddings USE"
    ):
        batch = textos[i:i + batch_size]

        batch_embeddings = model(batch).numpy()

        embeddings.append(batch_embeddings)

    return np.vstack(embeddings)

### **5. Geração dos embeddings do treino**

In [ ]:
X_train_embeddings_use = gerar_embeddings_use(
    X_train,
    model_use,
    batch_size=32
)

print(X_train_embeddings_use.shape)

Gerando embeddings USE:   0%|          | 0/503 [00:00<?, ?it/s]

(16073, 512)


### **6. Geração dos embeddings da validação**

In [ ]:
X_val_embeddings_use = gerar_embeddings_use(
    X_val,
    model_use,
    batch_size=32
)

print(X_val_embeddings_use.shape)

Gerando embeddings USE:   0%|          | 0/126 [00:00<?, ?it/s]

(4019, 512)


### **7. Criação do pipeline**

In [ ]:
pipeline_use_svm = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("clf", SVC(class_weight="balanced"))
    ]
)

### **8. Treinamento do modelo**

In [ ]:
pipeline_use_svm.fit(
    X_train_embeddings_use,
    y_train
)

Pipeline(steps=[('scaler', StandardScaler()),
                ('clf', SVC(class_weight='balanced'))])

### **9. Fazer as previsões**

In [ ]:
y_pred_use_svm = pipeline_use_svm.predict(
    X_val_embeddings_use
)

accuracy_use_svm = accuracy_score(
    y_val,
    y_pred_use_svm
)

macro_f1_use_svm = f1_score(
    y_val,
    y_pred_use_svm,
    average="macro"
)

print(f"Accuracy: {accuracy_use_svm:.4f}")
print(f"Macro-F1: {macro_f1_use_svm:.4f}")
print(classification_report(y_val,y_pred_use_svm))

Accuracy: 0.4454
Macro-F1: 0.4431
              precision    recall  f1-score   support

          c1       0.46      0.51      0.48      1269
        c234       0.39      0.33      0.36      1371
          c5       0.48      0.50      0.49      1379

    accuracy                           0.45      4019
   macro avg       0.44      0.45      0.44      4019
weighted avg       0.44      0.45      0.44      4019



# **Testando variações**

### **1. Fazendo GridSearch para modelo SVM**

In [ ]:
# from sklearn.svm import LinearSVC

# pipeline_use_linear = Pipeline(
#     steps=[
#         ("scaler", StandardScaler()),
#         ("clf", LinearSVC(
#             class_weight="balanced",
#             max_iter=10000
#         ))
#     ]
# )

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_use_svm = [
    {
        # "clf__kernel": ["linear"],
        "clf__C": [0.01, 0.1, 1, 10, 100]
    },
    # {
    #     "clf__kernel": ["rbf"],
    #     "clf__C": [0.01, 0.1, 1, 10, 100],
    #     "clf__gamma": ["scale", "auto"]
    # }
]

# Configura o GridSearch
grid_use_svm = GridSearchCV(
    estimator=pipeline_use_svm,
    param_grid=param_grid_use_svm,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    return_train_score=True,
    verbose=3
)

# param_grid_use_linear = {
#     "clf__C": [0.001, 0.01, 0.1, 1, 10, 100]
# }

# grid_use_linear = GridSearchCV(
#     estimator=pipeline_use_linear,
#     param_grid=param_grid_use_linear,
#     scoring="accuracy",
#     cv=5,
#     n_jobs=-1,
#     return_train_score=True,
#     verbose=3
# )

### **1.1 Executa o grid**

In [23]:
grid_use_svm.fit(
    X_train_embeddings_use,
    y_train
)

print("Melhores parâmetros:")
print(grid_use_svm.best_params_)

print("\nMelhor Accuracy média no CV:")
print(grid_use_svm.best_score_)

# grid_use_linear.fit(
#     X_train_embeddings_use,
#     y_train
# )

# print("Melhores parâmetros:")
# print(grid_use_linear.best_params_)

# print("\nMelhor Accuracy média no CV:")
# print(grid_use_linear.best_score_)

Fitting 5 folds for each of 5 candidates, totalling 25 fits


KeyboardInterrupt: 

In [ ]:
y_pred_use_grid = grid_use_svm.predict(
    X_val_embeddings_use
)

accuracy_use_grid = accuracy_score(
    y_val,
    y_pred_use_grid
)

macro_f1_use_grid = f1_score(
    y_val,
    y_pred_use_grid,
    average="macro"
)

print(f"Accuracy: {accuracy_use_grid:.4f}")
print(f"Macro-F1: {macro_f1_use_grid:.4f}")

print(
    classification_report(
        y_val,
        y_pred_use_grid
    )
)

# y_pred_use_linear = grid_use_linear.predict(
#     X_val_embeddings_use
# )

# accuracy_use_linear = accuracy_score(
#     y_val,
#     y_pred_use_linear
# )

# macro_f1_use_linear = f1_score(
#     y_val,
#     y_pred_use_linear,
#     average="macro"
# )

# print(f"Accuracy: {accuracy_use_linear:.4f}")
# print(f"Macro-F1: {macro_f1_use_linear:.4f}")

# print("\nClassification Report:")
# print(
#     classification_report(
#         y_val,
#         y_pred_use_linear
#     )
# )